# NOTE
- Using the extracted netlist (`extracted_netlist.spice`)

In [3]:
from utils.netlist_to_verilog import *
in_path, out_path = 'inputs/extracted_netlist.spice', 'inputs/puzzle.v'
forced_top = None

lib_ports, subckt_bodies, instantiated_types = parse_spice(in_path)
top_name = find_top(lib_ports, instantiated_types, forced_top)
top_ports = lib_ports[top_name]
top_ports = [p for p in top_ports if p not in POWER_PINS]

instances = parse_instances(subckt_bodies[top_name])

instances, clk_alias_map, removed = collapse_clock_trees(lib_ports, instances)
if removed:
    print(f"Collapsed {len(removed)} clock-buffer cells into "
          f"{len(set(clk_alias_map.values()))} canonical clock net(s): "
          f"{sorted(set(clk_alias_map.values()))}")

assigns, flops, ties = build_model(lib_ports, instances)
sanity_check(assigns, flops, ties)

verilog = generate_verilog(top_name, top_ports, assigns, flops, ties)
with open(out_path, "w") as f:
    f.write(verilog)

print(f"Top module   : {top_name}")
print(f"Instances    : {len(instances)}")
print(f"Comb. gates  : {len(assigns)}")
print(f"Flip-flops   : {len(flops)}")
print(f"Tie cells    : {len(ties)}")
print(f"Written to   : {out_path}")



Collapsed 32 clock-buffer cells into 1 canonical clock net(s): ['clk']
Top module   : puzzle
Instances    : 910
Comb. gates  : 598
Flip-flops   : 92
Tie cells    : 12
Written to   : inputs/puzzle.v
